In [1]:
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import os


In [3]:
#charger population
df_pop = pd.read_excel("data/raw/Population_communes.xlsx", skiprows = 7, sheet_name = "Communes")

#charger degré de densité de population en 2022
df_densite = pd.read_excel("data/raw/Degre_densite.xlsx", skiprows = 4)

#Aperçu des données
print("Degré de densité:")
display(df_densite.head())

print("Population:")
display(df_pop.head())


Degré de densité:


,CODGEO,LIBGEO,DENS,LIBDENS,PMUN22,P1,P2,P3,DENS_AAV,LIBDENS_AAV,DENS7,LIBDENS7
0,01001,L'Abergement-Clémenciat,3,Rural,859,0.0,0.000000,100.000000,4,Rural non périurbain,6,Rural à habitat dispersé
1,01002,L'Abergement-de-Varey,3,Rural,273,0.0,0.000000,100.000000,4,Rural non périurbain,6,Rural à habitat dispersé
2,01004,Ambérieu-en-Bugey,2,Urbain intermédiaire,15554,0.0,96.933762,3.066238,2,Urbain intermédiaire,2,Centres urbains intermédiaires
3,01005,Ambérieux-en-Dombes,3,Rural,1917,0.0,0.000000,100.000000,3,Rural périurbain,5,Bourgs ruraux
4,01006,Ambléon,3,Rural,114,0.0,0.000000,100.000000,4,Rural non périurbain,6,Rural à habitat dispersé


Population:


,Code région,Nom de la région,Code département,Code arrondissement,Code canton,Code commune,Nom de la commune,Population municipale,Population comptée à part,Population totale
0,84,Auvergne-Rhône-Alpes,01,2.0,08,1,L' Abergement-Clémenciat,860,16,876
1,84,Auvergne-Rhône-Alpes,01,1.0,01,2,L' Abergement-de-Varey,270,6,276
2,84,Auvergne-Rhône-Alpes,01,1.0,01,4,Ambérieu-en-Bugey,15934,405,16339
3,84,Auvergne-Rhône-Alpes,01,2.0,22,5,Ambérieux-en-Dombes,1906,24,1930
4,84,Auvergne-Rhône-Alpes,01,1.0,04,6,Ambléon,115,0,115


In [4]:
print(df_densite.columns)
print(df_pop.columns)

Index(['CODGEO', 'LIBGEO', 'DENS', 'LIBDENS', 'PMUN22', 'P1', 'P2', 'P3',
       'DENS_AAV', 'LIBDENS_AAV', 'DENS7', 'LIBDENS7'],
      dtype='object')
Index(['Code région', 'Nom de la région', 'Code département',
       'Code arrondissement', 'Code canton', 'Code commune',
       'Nom de la commune', 'Population municipale',
       'Population comptée à part', 'Population totale'],
      dtype='object')


In [5]:
#sélection des colonnes les plus intéressantes dans le degré de densité

df_densite_clean = df_densite[["CODGEO", "LIBDENS", "LIBDENS_AAV", "LIBDENS7"]].copy()

df_densite_clean.columns = ["code_commune", "degre_densite", "degre_densite_AAV", "degre_densite_7"]

df_densite_clean.columns
print(df_densite_clean.head())

  code_commune         degre_densite     degre_densite_AAV  \
0        01001                 Rural  Rural non périurbain   
1        01002                 Rural  Rural non périurbain   
2        01004  Urbain intermédiaire  Urbain intermédiaire   
3        01005                 Rural      Rural périurbain   
4        01006                 Rural  Rural non périurbain   

                  degre_densite_7  
0        Rural à habitat dispersé  
1        Rural à habitat dispersé  
2  Centres urbains intermédiaires  
3                   Bourgs ruraux  
4        Rural à habitat dispersé  


In [6]:
#enregistrement de la nouvelle base de densité dans le dossier processed
# Définition du chemin de destination
target_path = "data/processed/df_densite_clean.xlsx"

# Vérification/Création du dossier de destination
os.makedirs(os.path.dirname(target_path), exist_ok=True)

# Enregistrement en format Excel
df_densite_clean.to_excel(target_path, index=False, sheet_name='Densite_Nettoyee')

print(f"Fichier sauvegardé dans : {target_path}")

Fichier sauvegardé dans : data/processed/df_densite_clean.xlsx


In [7]:
#lecture du fichier csv des élections
df_elecsv = pd.read_csv("data/raw/resultats_definitifs_par_communes.csv", sep=";", low_memory=False)
print(df_elecsv.head())

  Code département Libellé département Code commune          Libellé commune  \
0                1                 Ain         1001  L'Abergement-Clémenciat   
1                1                 Ain         1002    L'Abergement-de-Varey   
2                1                 Ain         1004        Ambérieu-en-Bugey   
3                1                 Ain         1005      Ambérieux-en-Dombes   
4                1                 Ain         1006                  Ambléon   

   Inscrits  Votants % Votants  Abstentions % Abstentions  Exprimés  ...  \
0       662      492    74,32%          170        25,68%       476  ...   
1       228      178    78,07%           50        21,93%       171  ...   
2      8744     6037    69,04%         2707        30,96%      5890  ...   
3      1337      960    71,80%          377        28,20%       941  ...   
4        98       68    69,39%           30        30,61%        65  ...   

  Elu 203 Numéro de panneau 204  Nuance candidat 204 Nom candi

In [8]:

#visualisation des colonnes vides

df_elecsv.isna().sum()


Code département           0
Libellé département        0
Code commune               0
Libellé commune            0
Inscrits                   0
                       ...  
Sexe candidat 204      35231
Voix 204               35231
% Voix/inscrits 204    35231
% Voix/exprimés 204    35231
Elu 204                35232
Length: 1854, dtype: int64

In [9]:
#création de la colonne code_commune avec 5 chiffres

df_ele_clean = df_elecsv.copy()
df_ele_clean['code_commune'] = df_elecsv['Code commune'].astype(str).str.zfill(5)
df_ele_clean = df_ele_clean.drop('Code commune', axis=1)

#je replace la nouvelle colonne code_commune en 3ème position comme l'ancienne

cols = list(df_ele_clean.columns)
cols.remove('code_commune')
cols.insert(2, 'code_commune')
df_ele_clean = df_ele_clean[cols]
print(df_ele_clean.head())

  Code département Libellé département code_commune          Libellé commune  \
0                1                 Ain        01001  L'Abergement-Clémenciat   
1                1                 Ain        01002    L'Abergement-de-Varey   
2                1                 Ain        01004        Ambérieu-en-Bugey   
3                1                 Ain        01005      Ambérieux-en-Dombes   
4                1                 Ain        01006                  Ambléon   

   Inscrits  Votants % Votants  Abstentions % Abstentions  Exprimés  ...  \
0       662      492    74,32%          170        25,68%       476  ...   
1       228      178    78,07%           50        21,93%       171  ...   
2      8744     6037    69,04%         2707        30,96%      5890  ...   
3      1337      960    71,80%          377        28,20%       941  ...   
4        98       68    69,39%           30        30,61%        65  ...   

  Elu 203 Numéro de panneau 204  Nuance candidat 204 Nom candi